# Python for Machine Learning - Practice Notebook

**Kajola Gbenga**
Data Scientist & AI Engineer

*Companion executable notebook to the Python for Machine Learning handbook (Module 2). Every technique here is fully runnable - NumPy, pandas, and scikit-learn are all real, live libraries in this environment, so nothing is simulated.*

Run all cells top to bottom.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import time
import logging
import joblib

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("notebook")

print("NumPy:", np.__version__, "| pandas:", pd.__version__)

## 1. Generate the Realistic Customer Churn Dataset (Handbook Section 16)

A synthetic but realistic dataset with a deliberately messy `total_charges` column (stored as text, with blanks) - exactly the kind of real-world messiness Section 10 teaches you to handle.

In [ ]:
rng = np.random.default_rng(42)   # Section 6.7's modern Generator API
n = 1000

contract_types = rng.choice(["Month-to-month", "One year", "Two year"], size=n, p=[0.5, 0.3, 0.2])
payment_methods = rng.choice(["Electronic check", "Mailed check", "Bank transfer", "Credit card"], size=n)
internet = rng.choice(["DSL", "Fiber optic", "No"], size=n, p=[0.35, 0.45, 0.2])

tenure = rng.integers(0, 72, size=n)
age = rng.integers(18, 80, size=n)
monthly_charges = np.round(rng.uniform(20, 120, size=n), 2)
total_charges = np.round(monthly_charges * tenure + rng.normal(0, 50, size=n), 2)
total_charges = np.clip(total_charges, 0, None)
support_calls = rng.poisson(1.5, size=n)

churn_prob = 0.1 + 0.3*(contract_types == "Month-to-month") + 0.15*(support_calls > 3) + 0.1*(tenure < 6) - 0.1*(contract_types == "Two year")
churn_prob = np.clip(churn_prob, 0.02, 0.9)
churn = rng.binomial(1, churn_prob)

df = pd.DataFrame({
    "customer_id": [f"CUST-{i:05d}" for i in range(n)],
    "age": age, "gender": rng.choice(["Male", "Female"], size=n), "tenure": tenure,
    "monthly_charges": monthly_charges, "contract_type": contract_types, "payment_method": payment_methods,
    "support_calls": support_calls, "internet_service": internet,
    "total_charges": total_charges.astype(str), "churn": churn,
})
idx_missing = rng.choice(n, size=20, replace=False)
df.loc[idx_missing, "total_charges"] = " "   # deliberately messy, like real source data

df.to_csv("customer_churn.csv", index=False)
print(df.shape)
df.head()

## 2. NumPy Fundamentals: Scalars, Vectors, Matrices (Handbook Section 6)

In [ ]:
scalar = np.array(5)
vector = np.array([1, 2, 3])
matrix = np.array([[1, 2], [3, 4], [5, 6]])

for name, arr in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(f"{name}: ndim={arr.ndim}, shape={arr.shape}, size={arr.size}, dtype={arr.dtype}")

X = np.arange(12).reshape(3, 4)
print("\nOriginal 3x4 matrix:\n", X)
print("First row:", X[0])
print("Second column:", X[:, 1])
print("Reshaped to 4x3:\n", X.reshape(4, 3))

## 3. Broadcasting and Vectorization Speed (Handbook Sections 6.5, 7.1)

Same computation, two ways - a pure Python loop vs a vectorized NumPy operation.

In [ ]:
n = 2_000_000
values = np.random.default_rng(42).random(n)

start = time.perf_counter()
squared_loop = [v ** 2 for v in values]
loop_time = time.perf_counter() - start

start = time.perf_counter()
squared_vector = values ** 2
vector_time = time.perf_counter() - start

print(f"Loop:       {loop_time:.4f}s")
print(f"Vectorized: {vector_time:.4f}s")
print(f"Speedup:    {loop_time / vector_time:.0f}x")
assert list(squared_loop[:5]) == list(squared_vector[:5])   # confirm they compute the SAME thing

In [ ]:
# Broadcasting: a matrix minus its own per-row mean, no explicit loop needed
matrix = np.array([[1, 2, 3], [4, 5, 6], [10, 20, 30]])
row_means = matrix.mean(axis=1, keepdims=True)
centered = matrix - row_means
print("Row means:\n", row_means)
print("Centered matrix:\n", centered)
print("Each row's new mean (should be ~0):", centered.mean(axis=1))

## 4. Pandas Inspection and Cleaning (Handbook Sections 9-10)

In [ ]:
df = pd.read_csv("customer_churn.csv")

print(df.info())
print("\nMissing values per column:\n", df.isna().sum())
print("\nChurn class balance:\n", df["churn"].value_counts(normalize=True))

In [ ]:
# Clean: fix the messy total_charges column
print("Before cleaning, total_charges dtype:", df["total_charges"].dtype)
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce")
print("Blank strings became NaN:", df["total_charges"].isna().sum(), "rows")

df["total_charges"] = df["total_charges"].fillna(df["total_charges"].median())
df = df.drop_duplicates()
print("After cleaning, total_charges dtype:", df["total_charges"].dtype)
assert df["total_charges"].isna().sum() == 0

## 5. Data Leakage: Wrong vs Correct Order (Handbook Section 11)

Scaling before splitting lets the test set's own values influence the scaling statistic - here we show the scaler's learned mean/std actually DIFFER depending on whether the split happened first.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["customer_id", "churn"])
y = df["churn"]

# WRONG: compute the scaling statistic on the FULL dataset before splitting
full_mean = X["monthly_charges"].mean()
full_std = X["monthly_charges"].std()

# CORRECT: split FIRST, then compute the statistic on X_train ONLY
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
train_mean = X_train["monthly_charges"].mean()
train_std = X_train["monthly_charges"].std()

print(f"Mean computed on FULL dataset (LEAKS test data):  {full_mean:.4f}")
print(f"Mean computed on TRAIN split ONLY (correct):       {train_mean:.4f}")
print(f"Difference: {abs(full_mean - train_mean):.4f}")
print("\nEven a small difference here means every scaled feature value in a leaky pipeline")
print("is SLIGHTLY different from a correct pipeline - the model implicitly 'saw' test data.")

## 6. A Custom Scikit-Learn Transformer + Leakage-Safe Pipeline (Handbook Sections 5, 15, 50)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

class RatioFeatureAdder(BaseEstimator, TransformerMixin):
    """Adds a 'charges_per_tenure_month' feature - a custom, scikit-learn-compatible transformer."""
    def fit(self, X, y=None):
        return self   # stateless - nothing to learn from training data
    def transform(self, X):
        X = X.copy()
        X["charges_per_tenure_month"] = X["total_charges"] / X["tenure"].replace(0, 1)
        return X

numeric_features = ["age", "tenure", "monthly_charges", "support_calls", "total_charges", "charges_per_tenure_month"]
categorical_features = ["gender", "contract_type", "payment_method", "internet_service"]

numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

full_pipeline = Pipeline([
    ("add_ratio_feature", RatioFeatureAdder()),
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42)),
])

full_pipeline.fit(X_train, y_train)   # every fit() call inside this pipeline sees ONLY X_train
print("Pipeline fitted successfully - preprocessing and model trained together, leakage-safe by construction.")

## 7. Evaluation Beyond Raw Accuracy (Handbook Section 16.4)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

predictions = full_pipeline.predict(X_test)
probabilities = full_pipeline.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, predictions):.3f}")
print(f"Precision: {precision_score(y_test, predictions):.3f}")
print(f"Recall:    {recall_score(y_test, predictions):.3f}")
print(f"F1:        {f1_score(y_test, predictions):.3f}")
print("\nConfusion matrix:\n", confusion_matrix(y_test, predictions))
print("\n", classification_report(y_test, predictions))
print("Notice: accuracy alone would hide how few actual churners are being caught (low recall) -")
print("exactly Section 16.4's lesson about class imbalance.")

## 8. Model Persistence and Prediction on New Data (Handbook Sections 28, 42)

In [ ]:
joblib.dump(full_pipeline, "churn_model.joblib")

loaded_model = joblib.load("churn_model.joblib")
new_customer = pd.DataFrame([{
    "age": 29, "gender": "Male", "tenure": 2, "monthly_charges": 95.0,
    "contract_type": "Month-to-month", "payment_method": "Electronic check",
    "support_calls": 5, "internet_service": "Fiber optic", "total_charges": 190.0,
}])
prediction = loaded_model.predict(new_customer)[0]
probability = loaded_model.predict_proba(new_customer)[0][1]
print(f"Prediction: {'Will churn' if prediction == 1 else 'Will not churn'} (probability: {probability:.1%})")
print("\nNotice: new_customer needed NO manual preprocessing - the saved pipeline")
print("(including our custom RatioFeatureAdder) handled it automatically.")

## 9. The Standard ML Debugging Checklist, Live (Handbook Section 37)

Reproducing a real, common ML error on purpose, then applying the fix - passing a single extracted row as a `Series` instead of a 1-row `DataFrame`.

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_train dtypes:\n", X_train.dtypes)
print("\ny_train value counts:\n", y_train.value_counts())

In [ ]:
# Reproduce a classic "1D vs 2D" mistake: extracting a single row with single brackets
# returns a Series, not a DataFrame - breaking any step that expects DataFrame column access
single_row_wrong = X_test.iloc[0]   # a Series - effectively 1-D, columns become the INDEX instead
try:
    full_pipeline.predict(single_row_wrong)
except Exception as e:
    print(f"Reproduced error: {type(e).__name__}: {e}")
    print("\nThis happened inside our own RatioFeatureAdder transformer: X['tenure'] on a Series")
    print("returns a single scalar value, not a column - so .replace(0, 1) fails immediately.")
    print("This is the SAME root cause Section 37.3 warns about (a Series where a DataFrame was")
    print("expected) - scikit-learn's own bare '2D array' check would catch the same mistake")
    print("slightly differently on a pipeline without a custom transformer in front of it.")

# Fix: double brackets keep it a 1-row DataFrame (2-D), with real column access
single_row_correct = X_test.iloc[[0]]
result = full_pipeline.predict(single_row_correct)
print(f"\nFixed - prediction succeeded: {result}")

## 10. Performance Lab: apply() vs Vectorized (Handbook Sections 10.4, 54)

Two regimes: a mild single-column `.apply()` penalty, and a catastrophic `axis=1` penalty.

In [ ]:
bench_df = pd.DataFrame({"age": np.random.default_rng(1).integers(18, 80, size=500_000)})

start = time.perf_counter()
bench_df["bucket_apply"] = bench_df["age"].apply(lambda a: "young" if a < 30 else "senior" if a > 60 else "middle")
t_apply = time.perf_counter() - start

start = time.perf_counter()
conditions = [bench_df["age"] < 30, bench_df["age"] > 60]
bench_df["bucket_vec"] = np.select(conditions, ["young", "senior"], default="middle")
t_vec = time.perf_counter() - start

print(f"Single-column apply: {t_apply:.4f}s | vectorized: {t_vec:.4f}s | speedup: {t_apply/t_vec:.1f}x")
assert (bench_df["bucket_apply"] == bench_df["bucket_vec"]).all()

In [ ]:
# The catastrophic case: row-wise apply(axis=1) across two columns
bench_df2 = pd.DataFrame({
    "total_charges": np.random.default_rng(2).uniform(10, 500, size=200_000),
    "tenure": np.random.default_rng(3).integers(0, 72, size=200_000),
})

start = time.perf_counter()
bench_df2["ratio_apply"] = bench_df2.apply(
    lambda row: row["total_charges"] / row["tenure"] if row["tenure"] != 0 else row["total_charges"], axis=1
)
t_apply_axis1 = time.perf_counter() - start

start = time.perf_counter()
bench_df2["ratio_vec"] = bench_df2["total_charges"] / bench_df2["tenure"].replace(0, 1)
t_vec2 = time.perf_counter() - start

print(f"apply(axis=1):        {t_apply_axis1:.4f}s")
print(f"vectorized arithmetic: {t_vec2:.4f}s")
print(f"speedup:                {t_apply_axis1/t_vec2:.0f}x")
print("\nThis is why Section 54 calls axis=1 apply the single most dangerous performance mistake in pandas.")

## 11. Confusion Matrix and Metrics From Scratch, Verified Against scikit-learn (Handbook Section 51)

In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[tn, fp], [fn, tp]])

def precision_manual(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

manual_cm = confusion_matrix_manual(y_test, predictions)
sklearn_cm = confusion_matrix(y_test, predictions)
manual_precision = precision_manual(y_test, predictions)
sklearn_precision = precision_score(y_test, predictions)

print("Manual confusion matrix:\n", manual_cm)
print("scikit-learn confusion matrix:\n", sklearn_cm)
print("Match:", np.array_equal(manual_cm, sklearn_cm))
print(f"\nManual precision: {manual_precision:.4f} | scikit-learn precision: {sklearn_precision:.4f} | Match: {np.isclose(manual_precision, sklearn_precision)}")

## 12. Wrap Up

In [ ]:
print("All demonstrations completed successfully.")
print(f"Final model saved as: churn_model.joblib")
print(f"Dataset: {len(df)} customers, {y.mean():.1%} churn rate")
print("\nSee the full handbook (docx) for the complete 62-section reference, 50 exercises,")
print("25 coding challenges, 160 interview questions, and the full end-to-end capstone project.")